# Does a small model know what it doesn't know?

Calibration study of an open language model, 4-bit, on a single RTX 4070 (12 GB).

**This notebook currently covers Stage 1, step a1 (protocol §0-§1):** load each model in 4-bit and run one
forward pass. Confirm it fits in VRAM and that the logits tensor prints. Nothing else matters until this works.

Later steps (a2 dataset, a3 scoring function, a4 `scores.csv`, ...) are added below as they are built.

## §0 Environment check

The GPU must be visible to PyTorch, and `bitsandbytes` must be built against the right CUDA.

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows without Developer Mode: harmless cache warning

import torch

print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
assert torch.cuda.is_available(), "No CUDA device visible - install the cu124 build of torch (protocol §0)"

props = torch.cuda.get_device_properties(0)
print("GPU            :", props.name)
print("VRAM total     :", round(props.total_memory / 1e9, 1), "GB")

import transformers, bitsandbytes
print("transformers   :", transformers.__version__)
print("bitsandbytes   :", bitsandbytes.__version__)

torch          : 2.6.0+cu124
CUDA available : True
GPU            : NVIDIA GeForce RTX 4070
VRAM total     : 12.9 GB


C:\Users\billy\Documents\Artificial_Confidence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers   : 5.17.0
bitsandbytes   : 0.50.2


## §1 The two models

| Variant | Hugging Face repo | Notes |
|---|---|---|
| `stock` | [Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) | Official model. BF16 safetensors, about 16 GB. |
| `uncensored` | [huihui-ai/Huihui-Qwen3-8B-abliterated-v2](https://huggingface.co/huihui-ai/Huihui-Qwen3-8B-abliterated-v2) | Abliterated Qwen3-8B (refusal behaviour removed by editing weights, not by training). Same base, architecture and parameter count. BF16 safetensors, about 16 GB. |

Both are downloaded from the Hub on first use into `C:\Users\<you>\.cache\huggingface\hub` (about 32 GB
total) and quantised on the fly to nf4 with the **identical** `BitsAndBytesConfig`. GGUF files are **not** used:
the protocol needs raw logits from `transformers`.

**The two models are never in memory at the same time.** Each variant is loaded, scored, then freed before the
next one loads. Two reasons:

1. **It does not fit.** Each is roughly 5-6 GB at nf4, so two together would be about 11-12 GB on a 12 GB card that
   is also driving your display, with nothing left for activations.
2. **It is not needed.** Each variant's scores are computed independently and stored as rows in `scores.csv`
   (the `model` column). The stock-versus-uncensored comparison (struggle S4) is made afterwards, from the CSV.

Everything except `MODELS[variant]` is identical between the two runs, so any difference between conditions
is not caused by loading.

In [2]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELS = {
    "stock":      "Qwen/Qwen3-8B",
    "uncensored": "huihui-ai/Huihui-Qwen3-8B-abliterated-v2",
}
VARIANTS = ["stock", "uncensored"]      # run one or both; they always run one at a time
LETTERS = ["A", "B", "C", "D"]

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# One hard-coded question, only to confirm the model runs (the real dataset is step a2).
QUESTION = "Which of these is a source of light?"
OPTIONS  = ["The Moon", "A mirror", "The Sun", "A window"]


def build_prompt(tok, question, options):
    body = "\n".join(f"{L}. {t}" for L, t in zip(LETTERS, options))
    user = ("Answer the multiple-choice question with a single letter.\n\n"
            f"Question: {question}\n{body}")
    return tok.apply_chat_template(
        [{"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=False,           # Qwen3: suppress the <think> block
    ) + "Answer:"

## Step a1: load, one forward pass, free

`run_a1(variant)` loads one model, runs a single forward pass, prints the logits, and then **frees the GPU** before
returning. It hands back only plain Python values (no tensors), so nothing keeps the model alive.
`enable_thinking=False` stops Qwen3 opening a `<think>` block; if your installed `transformers` does not support it,
this is where it will show up.

In [3]:
def run_a1(variant):
    model_id = MODELS[variant]
    print("=" * 78)
    print(f"[{variant}]  {model_id}")
    print("=" * 78)

    torch.cuda.reset_peak_memory_stats()
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map="cuda:0",
    )
    model.eval()
    footprint = model.get_memory_footprint() / 1e9
    print(f"model footprint : {footprint:.2f} GB   (expect roughly 5-6 GB for an 8B model at nf4)")

    letter_ids = [tok.encode(" " + L, add_special_tokens=False)[-1] for L in LETTERS]
    prompt = build_prompt(tok, QUESTION, OPTIONS)
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits          # shape: [batch, sequence, vocab]

    last = logits[0, -1].float()
    print("logits tensor shape :", tuple(logits.shape))
    print("last-position logits:", last)

    top = torch.topk(torch.softmax(last, dim=-1), 5)
    top5 = [(tok.convert_ids_to_tokens(i), i, round(p, 4)) for p, i in zip(top.values.tolist(), top.indices.tolist())]
    print("top-5 next tokens (token, id, prob):")
    for t, i, p in top5:
        print(f"   {p:6.3f}  id={i:<7} {t!r}")

    peak = torch.cuda.max_memory_allocated() / 1e9
    result = dict(
        variant=variant, model_id=model_id, footprint_gb=round(footprint, 2), peak_vram_gb=round(peak, 2),
        logits_shape=tuple(logits.shape), letter_ids=letter_ids,
        letter_tokens=tok.convert_ids_to_tokens(letter_ids), prompt=prompt, top5=top5,
    )

    # free the GPU so the next variant loads into an empty card
    del model, inputs, logits, last, top
    gc.collect()
    torch.cuda.empty_cache()
    result["vram_after_free_gb"] = round(torch.cuda.memory_allocated() / 1e9, 2)
    print(f"peak VRAM       : {peak:.2f} GB of {props.total_memory / 1e9:.1f} GB")
    print(f"after freeing   : {result['vram_after_free_gb']:.2f} GB still allocated")
    return result


results = {}
for v in VARIANTS:                 # strictly sequential: one model in memory at a time
    results[v] = run_a1(v)

[stock]  Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/399 [00:01<08:56,  1.35s/it]

Loading weights:   1%|          | 2/399 [00:02<08:03,  1.22s/it]

Loading weights:   1%|          | 4/399 [00:02<03:14,  2.03it/s]

...

Loading weights:  97%|█████████▋| 389/399 [00:12<00:00, 37.42it/s]

Loading weights:  98%|█████████▊| 393/399 [00:12<00:00, 33.71it/s]

Loading weights: 100%|██████████| 399/399 [00:12<00:00, 31.30it/s]

model footprint : 5.96 GB   (expect roughly 5-6 GB for an 8B model at nf4)


logits tensor shape : (1, 54, 151936)
last-position logits: tensor([ -2.5625, -15.1250, -13.8125,  ...,  -5.4062,  -5.4062,  -5.4062],
       device='cuda:0')
top-5 next tokens (token, id, prob):
    1.000  id=3070    'Ġ**'
    0.000  id=356     'ĠC'
    0.000  id=57960   'Ġ$\\'
    0.000  id=1124    'Ġ\\'
    0.000  id=1304    'Ġ__'
peak VRAM       : 6.18 GB of 12.9 GB
after freeing   : 0.01 GB still allocated


## a1 summary and checks

In [4]:
import pandas as pd

summary = pd.DataFrame(
    [{k: r[k] for k in ("variant", "model_id", "footprint_gb", "peak_vram_gb", "vram_after_free_gb")} for r in results.values()]
).set_index("variant")
display(summary)

total = props.total_memory / 1e9
for r in results.values():
    assert r["peak_vram_gb"] < total * 0.95, f"{r['variant']}: too close to the VRAM limit - consider a 4B model (protocol §1)"
    if r["vram_after_free_gb"] > 1.0:
        print(f"WARNING [{r['variant']}]: {r['vram_after_free_gb']} GB still allocated after freeing - "
              "the next model may not fit. Restart the kernel and run one variant at a time.")

if len(results) == 2:
    a, b = results["stock"], results["uncensored"]
    print("letter token ids identical :", a["letter_ids"] == b["letter_ids"], a["letter_tokens"])
    print("chat-template prompt identical:", a["prompt"] == b["prompt"])
    assert a["letter_ids"] == b["letter_ids"] and a["prompt"] == b["prompt"], \
        "The two variants tokenise differently - the comparison would be confounded"

print("a1 done: each model loads in 4-bit, a forward pass runs, logits print, and the GPU is freed between models.")

,model_id,footprint_gb,peak_vram_gb,vram_after_free_gb
variant,,,,
stock,Qwen/Qwen3-8B,5.96,6.17,0.01
uncensored,huihui-ai/Huihui-Qwen3-8B-abliterated-v2,5.96,6.18,0.01


letter token ids identical : True ['ĠA', 'ĠB', 'ĠC', 'ĠD']
chat-template prompt identical: True
a1 done: each model loads in 4-bit, a forward pass runs, logits print, and the GPU is freed between models.


## What to check before moving on to a2

- Footprint is roughly 5-6 GB per model and peak VRAM is comfortably under 12 GB.
- The top next-token candidates look like an answer (a letter with a leading space, e.g. `'ĠC'`), not `<think>`
  or garbage. If you see `<think>`, upgrade `transformers` or append the template's empty-think marker manually.
- The "after freeing" number is close to 0 GB, so the second model really loaded into an empty card.
- The two variants agree on letter token ids and prompt text (asserted above).
- Note anything that broke (version drift in `transformers` / `bitsandbytes` is common) in `notes/breakage-log.md`.